# Chess.com Exporter

### Download Data from User Stats and Games Played Results and Export to CSV

----

API Docs:

* https://www.chess.com/club/chess-com-developer-community
* https://www.chess.com/news/view/published-data-api

-----

## Configure User Info

In [19]:
my_user = "GrantedV1rus"

------

## Libraries and Dependencies

In [20]:
import requests
import time
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd

-----

# Get User Info and Current Stats

In [21]:
def getChessUser(username):
    url = "https://api.chess.com/pub/player/" + username + "/"
    headers = {"User-Agent": "chess.com-exporter/1.0"}
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    return response.json()

In [22]:
user_info = getChessUser(my_user)
# print(json.dumps(user_info, indent=4, sort_keys=True))

In [23]:
joined_date = datetime.fromtimestamp(user_info['joined'], tz=timezone.utc)

print(joined_date)

2025-12-18 20:57:17+00:00


In [24]:
def getChessRatings(username):
    url = "https://api.chess.com/pub/player/" + username + "/stats"
    headers = {"User-Agent": "chess.com-exporter/1.0"}
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    return response.json()

In [25]:
user_stats = getChessRatings(my_user)
# print(json.dumps(user_stats, indent=4, sort_keys=True))

In [26]:
# latest blitz rating
print(user_stats['chess_blitz']['last'])

{'rating': 983, 'date': 1794334855, 'rd': 24}


-----

# Get Archives List

In [27]:
def getChessUserArchives(username):
    url = "https://api.chess.com/pub/player/" + username + "/games/archives"
    headers = {"User-Agent": "chess.com-exporter/1.0"}
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    return response.json()["archives"]

In [28]:
archives = getChessUserArchives(my_user)

In [29]:
# total archives / months with games
len(archives)

12

In [30]:
monthlist = []

for i in archives:
    full_archive_link = i
    month = full_archive_link.rstrip('/').rsplit('/games/', 1)[-1]
    monthlist.append(month)

In [31]:
# helper function
def months(start_month, start_year, end_month, end_year):

    monthlist = []
    month, year = end_month, end_year

    while (year, month) >= (start_year, start_month):

        str = format(year) + '/' + '{:02d}'.format(month)
        monthlist.append(str)

        month -= 1
        if month < 1:
            month = 12
            year -= 1
    
    return monthlist

In [32]:
# Alternative Option 2: Get all Months Since Joined
start_year = joined_date.year
start_month = joined_date.month

end_date = datetime.now(timezone.utc)
end_year = end_date.year
end_month = end_date.month

In [33]:
# Alternative Option 3: manually set dates

# start_year = 2018
# start_month = 1

# end_year = 2018
# end_month = 6

In [34]:
# uncoment to run option 2 or 3
# monthlist = months(start_month, start_year, end_month, end_year)

----

# Download Data on Chess.com Games Played

In [35]:
def getChessGamesPlayed(username, monthlist):
    games_list = []
    headers = {"User-Agent": "chess.com-exporter/1.0"}

    for archive_month in monthlist:
        print("Getting Game Data for " + archive_month)
        url = "https://api.chess.com/pub/player/" + username + "/games/" + archive_month + "/"
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        games_list.extend(response.json().get("games", []))

        time.sleep(1)

    return games_list

In [36]:
# get all games stats
games_list = getChessGamesPlayed(my_user, monthlist)

Getting Game Data for 2025/12
Getting Game Data for 2026/01
Getting Game Data for 2026/02
Getting Game Data for 2026/03
Getting Game Data for 2026/04
Getting Game Data for 2026/05
Getting Game Data for 2026/06
Getting Game Data for 2026/07
Getting Game Data for 2026/08
Getting Game Data for 2026/09
Getting Game Data for 2026/10
Getting Game Data for 2026/11


In [37]:
# function to unpack nested data
def unpack(df, column, fillna=None):
    nested = pd.DataFrame(df[column].tolist(), index=df.index)
    if fillna is not None:
        nested = nested.fillna(fillna)
    nested = nested.rename(columns={
        "rating": column + "_rating",
        "result": column + "_result",
        "@id": column + "_id",
        "username": column + "_username",
    })
    return pd.concat([df.drop(columns=[column]), nested], axis=1)

## Initial Games Played Data Cleanup

In [38]:
# unpack the nested data 
games_played = pd.DataFrame(games_list)

games_played = unpack(games_played, 'black', 0)
games_played = unpack(games_played, 'white', 0)

for timestamp_column in ['start_time', 'end_time']:
    if timestamp_column in games_played.columns:
        games_played[timestamp_column] = pd.to_datetime(
            games_played[timestamp_column], unit='s'
        )
    
# set which side I played
games_played['my_side'] = np.where(games_played['white_username'] == my_user, 'white', 'black')

# get and assign my result
game_results = []

for index, row in games_played.iterrows():
    if row['white_username'] == my_user:
        # print("I was white")
        game_results.append(row['white_result'])
    else:
        # print("I was black")
        game_results.append(row['black_result'])
        
games_played['my_result'] = game_results

In [39]:
# export to csv
from pathlib import Path
Path("data").mkdir(parents=True, exist_ok=True)
games_played.to_csv("data/" + my_user + "_games_played" + ".csv", index=False)

In [40]:
# total games played
total_games_played = len(games_played)

In [41]:
total_games_played

2974

-------

## Advanced PGN Parsing: Extract Additional Game Info from PGN Game Files

**NOTE:** To use this section, you need to install the Chess-Python Library, which you can do on the command line using the following command: 

> pip install python-chess

In [42]:
import chess.pgn
import io
import re

In [43]:
# Advanced Parsing of PGN

terminations = []
total_moves = []
start_times = []
end_times  = []
moves_times_white = []
moves_times_black = []

# TODO: Additional Data to Exract
# Result
# ECO
# ECOUrl
# WhiteElo
# BlackElo

for index, row in games_played.iterrows():
    raw_pgn = io.StringIO(row['pgn'])
    game = chess.pgn.read_game(raw_pgn)
    
    terminations.append(game.headers['Termination'] )

    start_time = game.headers['Date'] + " " + game.headers['StartTime']
    start_times.append(start_time)  

    end_time = game.headers['Date'] + " " + game.headers['EndTime']
    end_times.append(end_time) 
        
    move_times = re.findall(r'\{\[%clk (.*?)\]\}', row['pgn'])
    
    total_moves.append(len(move_times))
    
    # last move time: 
    # print(move_times[-1])
    
    moves_times_white.append(', '.join(move_times[0:][::2]))
    moves_times_black.append(', '.join(move_times[1:][::2]))

games_played['termination'] = terminations 
games_played['total_moves'] = total_moves                       
                       
games_played['start_time'] = start_times   
games_played['start_time'] = pd.to_datetime(games_played['start_time'])

games_played['pgn_end_time'] = end_times   
games_played['pgn_end_time'] = pd.to_datetime(games_played['pgn_end_time'])

games_played['moves_times_white'] = moves_times_white
games_played['moves_times_black'] = moves_times_black

In [44]:
# TODO: 
# * Get the total play time of each player from last timestamp minus total seconds of your side game
# * Calculate the average move time 

In [45]:
# games_played.tail()

In [46]:
# export detailed stats to csv
Path("data").mkdir(parents=True, exist_ok=True)
games_played.to_csv("data/" + my_user + "_games_played_detailed" + ".csv", index=False)